# Sistema multiagente de cruce vial — Av. Constitución × Av. Cuauhtémoc (Monterrey, N.L.)

**Materia:** Modelación de sistemas multiagente
**Caso:** Intersección en forma de `+` con 4 aproximaciones (Norte, Sur, Este, Oeste).

Este notebook implementa una simulación basada en agentes (*agent-based model*) de un cruce vial
real de Monterrey. Cada **vehículo** es un agente autónomo que avanza paso a paso por una
**cuadrícula discreta** que representa las dos avenidas. El control del cruce es **emergente**:
no hay semáforos ni prioridades centrales; cada auto solo avanza si su celda de enfrente está
libre y, al entrar al cruce, solo si puede atravesarlo completo. Con esas dos reglas locales se
evitan tanto las colisiones como el bloqueo total del cruce.

**Cómo usarlo en Google Colab:** ejecuta las celdas en orden (`Entorno de ejecución → Ejecutar todo`).
Todo el código es autocontenido y usa `random.seed(42)` para que los resultados sean reproducibles.

> **Pega en tu reporte:** la animación (Celda 7), la imagen del estado final (Celda 8) y la
> gráfica de barras de métricas (Celda 9).

In [ ]:
# === Celda 1 — Librerías y reproducibilidad ===
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Patch
from matplotlib import rc

# Semilla fija -> la simulacion da SIEMPRE el mismo resultado (reproducibilidad).
random.seed(42)
np.random.seed(42)

# Hace que FuncAnimation se muestre como reproductor HTML dentro de Colab/Jupyter
# (no requiere ffmpeg). Si lo prefieres como video usa anim.to_html5_video().
rc("animation", html="jshtml")

print("Librerias cargadas. Semilla fijada en 42.")

In [ ]:
# === Celda 2 — Definición del ENTORNO (la cuadrícula) ===
# El entorno es una rejilla discreta de GRID_SIZE x GRID_SIZE.
# - Constitucion es HORIZONTAL -> ocupa dos filas (un carril por sentido).
# - Cuauhtemoc  es VERTICAL    -> ocupa dos columnas (un carril por sentido).
# - El resto de las celdas son MANZANAS (no transitables).
# El cruce queda en el centro del grid (4 celdas centrales = zona de conflicto).

GRID_SIZE = 15                 # cuadricula 15x15 (>= 15 como pide el requisito)
CENTER    = GRID_SIZE // 2     # = 7 -> centro de la interseccion

# Cuauhtemoc (vertical): dos carriles adyacentes con sentidos opuestos.
COL_NS = CENTER + 1            # col 8: sentido Norte -> Sur  (los autos BAJAN)
COL_SN = CENTER               # col 7: sentido Sur -> Norte  (los autos SUBEN)

# Constitucion (horizontal): dos carriles adyacentes con sentidos opuestos.
ROW_OE = CENTER               # fila 7: sentido Oeste -> Este (van a la DERECHA)
ROW_EO = CENTER + 1           # fila 8: sentido Este -> Oeste (van a la IZQUIERDA)

STREET_ROWS = {ROW_OE, ROW_EO}            # filas que son calle (Constitucion)
STREET_COLS = {COL_SN, COL_NS}            # columnas que son calle (Cuauhtemoc)

# Las 4 celdas centrales donde ambas avenidas se cruzan (zona de conflicto).
CROSS_CELLS = {(r, c) for r in STREET_ROWS for c in STREET_COLS}

def is_drivable(row, col):
    """Una celda es transitable si pertenece a Constitucion (fila) o a Cuauhtemoc (col)."""
    return row in STREET_ROWS or col in STREET_COLS

# Matriz de fondo para dibujar: valor alto = calle (claro), valor bajo = manzana (oscuro).
grid_background = np.full((GRID_SIZE, GRID_SIZE), 0.18)   # manzanas (gris oscuro)
for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        if is_drivable(r, c):
            grid_background[r, c] = 0.78                  # calles (gris claro)

print(f"Grid {GRID_SIZE}x{GRID_SIZE}. Centro en fila/col {CENTER}.")
print(f"Calles -> filas {sorted(STREET_ROWS)} (Constitucion), cols {sorted(STREET_COLS)} (Cuauhtemoc).")
print(f"Celdas de cruce: {sorted(CROSS_CELLS)}")

In [ ]:
# === Celda 3 — Definición del AGENTE (clase Vehicle) ===
# Cada vehiculo es un agente con estado minimo: posicion, direccion y si sigue activo.
# Avanza en LINEA RECTA (no gira) hasta salir del grid.

# Color y etiqueta segun la APROXIMACION por la que entra el vehiculo.
DIRECTION_INFO = {
    ( 1, 0): {"approach": "Norte (baja)",      "color": "#e63946"},  # entra arriba, va al Sur
    (-1, 0): {"approach": "Sur (sube)",        "color": "#1d4ed8"},  # entra abajo, va al Norte
    ( 0, 1): {"approach": "Oeste (a la der.)", "color": "#2a9d8f"},  # entra izq., va al Este
    ( 0,-1): {"approach": "Este (a la izq.)",  "color": "#f4a261"},  # entra der., va al Oeste
}

class Vehicle:
    """Agente vehiculo. Ocupa UNA celda y avanza un paso por turno segun su direccion."""
    _id_counter = 0  # contador global para asignar un id unico a cada agente

    def __init__(self, row, col, direction):
        self.id = Vehicle._id_counter
        Vehicle._id_counter += 1
        self.row = row
        self.col = col
        self.direction = direction      # (drow, dcol): cuanto se mueve en fila y columna
        self.state = "moving"           # "moving" mientras esta en el grid, "done" al salir
        self.crossed = False            # True cuando ya paso por la zona central del cruce

    @property
    def pos(self):
        """Posicion actual como tupla (fila, col)."""
        return (self.row, self.col)

    def next_cell(self):
        """Celda a la que INTENTARIA moverse en el siguiente paso."""
        return (self.row + self.direction[0], self.col + self.direction[1])

    def move_to(self, row, col):
        """Mueve el agente a (row, col). Devuelve True si en este paso entro al cruce."""
        self.row, self.col = row, col
        if not self.crossed and self.pos in CROSS_CELLS:
            self.crossed = True
            return True
        return False

    @property
    def color(self):
        return DIRECTION_INFO[self.direction]["color"]

print("Clase Vehicle lista.")

In [ ]:
# === Celda 4 — Definición del ENTORNO dinámico (clase IntersectionEnv) ===
# Administra el grid, crea (spawn) vehiculos en las 4 entradas y actualiza el sistema
# en cada paso mediante step().

class IntersectionEnv:
    """Entorno que gestiona spawn de vehiculos, su actualizacion y la lista de activos."""

    # Puntos de entrada: (fila, col, direccion) para cada aproximacion.
    # Cada vehiculo entra en el borde del grid por el carril que le corresponde.
    SPAWN_POINTS = [
        (0,             COL_NS, ( 1, 0)),  # Norte: entra arriba y baja
        (GRID_SIZE - 1, COL_SN, (-1, 0)),  # Sur:   entra abajo y sube
        (ROW_OE,        0,      ( 0, 1)),  # Oeste: entra a la izquierda y va a la derecha
        (ROW_EO, GRID_SIZE - 1, ( 0,-1)),  # Este:  entra a la derecha y va a la izquierda
    ]

    def __init__(self, spawn_prob=0.25):
        self.spawn_prob    = spawn_prob  # probabilidad (por entrada y por paso) de crear un auto
        self.vehicles      = []          # lista de agentes ACTIVOS (state == "moving")
        self.total_spawned = 0           # cuantos vehiculos se han creado en total
        self.total_exited  = 0           # cuantos salieron del grid
        self.total_crossed = 0           # cuantos cruzaron la zona central (throughput)
        self.time          = 0           # numero de pasos simulados

    def occupied_cells(self):
        """Conjunto de celdas ocupadas ahora mismo (para checar colisiones en O(1))."""
        return {v.pos for v in self.vehicles if v.state == "moving"}

    def in_grid(self, row, col):
        return 0 <= row < GRID_SIZE and 0 <= col < GRID_SIZE

    def try_spawn(self, occupied):
        """Con probabilidad spawn_prob intenta crear un auto en cada entrada que este libre."""
        for row, col, direction in self.SPAWN_POINTS:
            if random.random() < self.spawn_prob and (row, col) not in occupied:
                self.vehicles.append(Vehicle(row, col, direction))
                occupied.add((row, col))      # se ocupa la celda de entrada de inmediato
                self.total_spawned += 1

    def box_clear(self, v, occupied):
        """Regla local 'NO bloquear el cruce' (como 'no te quedes atravesado en la interseccion').

        Si el auto va a ENTRAR al cruce viniendo de fuera, exige que TODA su trayectoria por
        las celdas centrales y la primera celda de SALIDA esten libres. Asi nunca se quedan
        cuatro autos atrapados rotando entre las 4 celdas del centro (bloqueo circular/gridlock).
        No es un semaforo ni una prioridad entre autos: cada agente lo decide solo, localmente.
        """
        nr, nc = v.next_cell()
        if (nr, nc) not in CROSS_CELLS or v.pos in CROSS_CELLS:
            return True                       # no aplica: no entra al cruce o ya esta dentro
        dr, dc = v.direction
        r, c = nr, nc
        while (r, c) in CROSS_CELLS:           # recorre las celdas del cruce en su direccion
            if (r, c) in occupied:
                return False
            r += dr; c += dc
        if self.in_grid(r, c) and (r, c) in occupied:   # celda de salida del cruce
            return False
        return True

    def step(self):
        """Un paso de tiempo: 1) intenta crear autos, 2) mueve a todos los agentes.

        Regla de movimiento: un auto avanza solo si la celda siguiente esta dentro del grid,
        es transitable y NO esta ocupada (y, si va a entrar al cruce, este no se bloqueara).
        Si no puede, ESPERA (nunca hay colisiones: una celda = a lo sumo un auto).

        Se resuelve por 'caravanas': se repiten pasadas hasta que nadie mas pueda avanzar,
        pero cada auto se mueve a lo sumo UNA celda por paso (marcado en moved_set). 'occupied'
        se ACTUALIZA al instante en que un auto se mueve, para que dos autos no entren a la
        misma celda en la misma pasada (esto es lo que evita que las 'bolitas' se traslapen).
        """
        self.time += 1
        occupied = self.occupied_cells()     # se calcula UNA vez y se mantiene actualizado
        self.try_spawn(occupied)             # 1) nacen autos nuevos en las entradas

        moved_set = set()                    # ids de autos que ya se movieron en ESTE paso
        progress = True
        while progress:                      # 2) pasadas hasta punto fijo
            progress = False
            for v in self.vehicles:
                if v.state != "moving" or v.id in moved_set:
                    continue
                nr, nc = v.next_cell()
                if not self.in_grid(nr, nc):         # la celda siguiente esta FUERA del grid
                    v.state = "done"                 # -> el auto sale y termina su recorrido
                    self.total_exited += 1
                    occupied.discard(v.pos)          # libera su celda
                    moved_set.add(v.id); progress = True
                    continue
                # avanza solo si la celda esta libre, es calle y no bloqueara el cruce
                if (is_drivable(nr, nc) and (nr, nc) not in occupied
                        and self.box_clear(v, occupied)):
                    occupied.discard(v.pos)          # libera la celda actual de INMEDIATO
                    if v.move_to(nr, nc):            # avanza una celda
                        self.total_crossed += 1      # contabiliza el cruce la primera vez
                    occupied.add(v.pos)              # ocupa la nueva celda de INMEDIATO
                    moved_set.add(v.id); progress = True
                # si no se cumple la condicion -> el auto espera (no se mueve, no choca)

        # se eliminan de la lista de activos los que salieron del grid
        self.vehicles = [v for v in self.vehicles if v.state == "moving"]

    def snapshot(self):
        """Foto del estado actual: lista de (fila, col, direccion) para animar despues."""
        return [(v.row, v.col, v.direction) for v in self.vehicles]

print("Clase IntersectionEnv lista.")

In [ ]:
# === Celda 5 — Función de dibujo y leyenda ===
# draw_state pinta el fondo (calles/manzanas) y los vehiculos como puntos de color.
# Se reutiliza tanto en la animacion como en la imagen del estado final.

def draw_state(ax, vehicles_state, title=""):
    ax.clear()
    # Fondo: manzanas oscuras, calles claras.
    ax.imshow(grid_background, cmap="Greys_r", vmin=0, vmax=1, origin="upper")
    # Vehiculos: un punto por agente, coloreado segun su aproximacion.
    for row, col, direction in vehicles_state:
        ax.scatter(col, row, c=DIRECTION_INFO[direction]["color"],
                   s=170, edgecolors="black", linewidths=0.6, zorder=3)
    ax.set_title(title)
    ax.set_xticks(np.arange(-0.5, GRID_SIZE, 1))
    ax.set_yticks(np.arange(-0.5, GRID_SIZE, 1))
    ax.set_xticklabels([]); ax.set_yticklabels([])
    ax.grid(True, color="white", linewidth=0.3, alpha=0.25)
    ax.set_xlim(-0.5, GRID_SIZE - 0.5)
    ax.set_ylim(GRID_SIZE - 0.5, -0.5)   # fila 0 arriba (orientacion tipo mapa)

# Handles de la leyenda (un color por aproximacion).
LEGEND_HANDLES = [Patch(facecolor=info["color"], edgecolor="black", label=info["approach"])
                  for info in DIRECTION_INFO.values()]

print("Funcion de dibujo lista.")

In [ ]:
# === Celda 6 — Correr la simulación (reproducible) ===
SPAWN_PROB = 0.25   # probabilidad de aparicion por entrada y por paso (sube/baja la densidad)
STEPS      = 70     # numero de pasos de tiempo a simular

# Reinicio limpio para que la corrida sea 100% reproducible aunque se re-ejecute la celda.
Vehicle._id_counter = 0
random.seed(42)

env = IntersectionEnv(spawn_prob=SPAWN_PROB)
history = [env.snapshot()]                 # guardamos el estado de cada paso para animarlo
for _ in range(STEPS):
    env.step()
    history.append(env.snapshot())

print(f"Pasos simulados                      : {env.time}")
print(f"Vehiculos creados                    : {env.total_spawned}")
print(f"Cruzaron la interseccion (throughput): {env.total_crossed}")
print(f"Salieron del grid                    : {env.total_exited}")
print(f"Aun en transito al final             : {len(env.vehicles)}")

In [ ]:
# === Celda 7 — ANIMACIÓN (FuncAnimation) ===
# Reproduce el historial guardado. Colores por direccion + leyenda por aproximacion.
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    draw_state(ax, history[frame],
               title=f"Cruce Constitucion x Cuauhtemoc  -  paso {frame}")
    ax.legend(handles=LEGEND_HANDLES, loc="upper left",
              bbox_to_anchor=(1.02, 1.0), fontsize=8, title="Aproximacion")
    return ax

anim = animation.FuncAnimation(fig, update, frames=len(history),
                               interval=300, blit=False)
plt.close(fig)   # evita que se muestre una figura estatica extra debajo del reproductor
anim             # en Colab/Jupyter esto muestra el reproductor de la animacion

In [ ]:
# === Celda 8 — Gráfica estática del estado final ===
fig2, ax2 = plt.subplots(figsize=(6, 6))
draw_state(ax2, history[-1], title=f"Estado final (paso {len(history) - 1})")
ax2.legend(handles=LEGEND_HANDLES, loc="upper left",
           bbox_to_anchor=(1.02, 1.0), fontsize=8, title="Aproximacion")
plt.tight_layout()
plt.show()

## Adaptación del ejemplo al cruce real y simplificaciones

Se adaptó un esquema genérico de simulación basada en agentes al cruce de **Av. Constitución
(horizontal) y Av. Cuauhtémoc (vertical)** en Monterrey, modelándolo como una cuadrícula `15×15`
donde solo las dos avenidas son transitables y todo lo demás son **manzanas** no transitables.
La intersección en `+` se ubica al centro y cada avenida se representó con **dos carriles de sentido
opuesto** (circulación por la derecha), de modo que aparecen las 4 aproximaciones: Norte, Sur,
Este y Oeste. Los **vehículos** son agentes que entran por los bordes, avanzan en **línea recta**
un paso por turno y desaparecen al salir del grid.

**Qué se simplificó:** (1) no hay semáforos ni reglas de prioridad entre autos; el control es
**local**: cada agente avanza solo si su celda de enfrente está libre y, si va a entrar al cruce,
solo si puede atravesarlo completo (regla "no bloquear la intersección", como no quedarse
atravesado en un cruce real). Gracias a esto, evitar choques y evitar el bloqueo total
(*gridlock*) son comportamientos **emergentes**, sin un controlador central. (2) Los vehículos no
giran ni cambian de carril (trayectorias rectas); (3) todos viajan a la misma velocidad (una celda
por paso); (4) se ignora la geometría real, longitudes de calle, peatones y el tamaño físico del
auto (cada uno ocupa una sola celda). Estas decisiones mantienen el modelo **mínimo y entendible**,
suficiente para observar la formación de colas y el flujo (*throughput*) a través del cruce.

In [ ]:
# === Celda 9 (EXTRA) — Métricas y gráfica de barras ===
labels = ["Creados", "Cruzaron\n(throughput)", "Salieron\ndel grid"]
values = [env.total_spawned, env.total_crossed, env.total_exited]
colors = ["#264653", "#2a9d8f", "#e9c46a"]

fig3, ax3 = plt.subplots(figsize=(6, 4))
bars = ax3.bar(labels, values, color=colors, edgecolor="black")
for b, val in zip(bars, values):                     # etiqueta el valor encima de cada barra
    ax3.text(b.get_x() + b.get_width() / 2, val + 0.4, str(val),
             ha="center", va="bottom", fontweight="bold")
ax3.set_ylabel("Numero de vehiculos")
ax3.set_title("Metricas de la simulacion")
ax3.set_ylim(0, max(values) + 4)
plt.tight_layout()
plt.show()

tasa = (env.total_crossed / env.total_spawned * 100) if env.total_spawned else 0
print(f"Tasa de cruce: {tasa:.1f}% de los vehiculos creados llegaron a cruzar la interseccion.")
print(f"Vehiculos aun en transito al terminar la simulacion: {len(env.vehicles)}")